In [27]:
import json
from typing import Dict, Optional


def average_iter_metrics_for_batch(
    batch_size: int,
    file_path: str,
    tolerance: int = 4,
) -> Dict[str, Optional[float]]:
    """
    Compute average timing metrics from rows where BOTH:
    - abs(row['batch_size'] - batch_size) <= tolerance
    - abs(row['num_scheduled_tokens'] - batch_size) <= tolerance

    Returns averages for:
    latency_ms, model_compute_ms, expert_ffn_ms, attention_ms.
    Values are None if no numeric values are found for that metric.
    """
    metric_names = [
        "latency_ms",
        "model_compute_ms",
        "expert_ffn_ms",
        "attention_ms",
    ]

    sums = {metric: 0.0 for metric in metric_names}
    counts = {metric: 0 for metric in metric_names}
    matched_rows = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            row = json.loads(line)

            row_batch_size = row.get("batch_size")
            row_tokens = row.get("num_scheduled_tokens")

            if row_batch_size is None or row_tokens is None:
                continue

            if (
                batch_size - row_batch_size <= tolerance
                and abs(batch_size - row_tokens) <= tolerance
            ):
                matched_rows += 1

                for metric in metric_names:
                    value = row.get(metric)
                    if isinstance(value, (int, float)):
                        sums[metric] += float(value)
                        counts[metric] += 1

    if matched_rows == 0:
        raise ValueError(
            "No rows matched. Check batch_size/tolerance, or verify the file content."
        )

    averages = {
        metric: (sums[metric] / counts[metric]) if counts[metric] else None
        for metric in metric_names
    }

    return {
        **averages,
        "matched_rows": float(matched_rows),
    }




In [28]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_40bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776647249336102073_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=40, file_path=sample_file_path)
result

{'latency_ms': 23.171858170838135,
 'model_compute_ms': 22.37037609290095,
 'expert_ffn_ms': 26.673177229192614,
 'attention_ms': 0.0,
 'matched_rows': 583.0}

In [29]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_16bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776647902399331951_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=16, file_path=sample_file_path)
result

{'latency_ms': 16.795476696473667,
 'model_compute_ms': 16.17129049137125,
 'expert_ffn_ms': 25.417028094363154,
 'attention_ms': 0.0,
 'matched_rows': 2035.0}

In [30]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_200bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776648129821772759_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=200, file_path=sample_file_path)
result

{'latency_ms': 36.960467176084165,
 'model_compute_ms': 33.20412411159939,
 'expert_ffn_ms': 27.045108747703058,
 'attention_ms': 6.1590153638963345,
 'matched_rows': 270.0}

In [31]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_140bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776648682840783640_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=140, file_path=sample_file_path)
result

{'latency_ms': 31.859840311971652,
 'model_compute_ms': 29.48852570202886,
 'expert_ffn_ms': 24.713860128302965,
 'attention_ms': 4.7913721861685215,
 'matched_rows': 588.0}

In [32]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_400bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776649034978024607_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=400, file_path=sample_file_path, tolerance=20)
result

{'latency_ms': 48.938224411010744,
 'model_compute_ms': 42.07099838256836,
 'expert_ffn_ms': 21.17774721980095,
 'attention_ms': 20.89325116276741,
 'matched_rows': 10.0}

In [33]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_312bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776649470343737480_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=312, file_path=sample_file_path, tolerance=20)
result

{'latency_ms': 45.720050511397716,
 'model_compute_ms': 40.24405195581632,
 'expert_ffn_ms': 25.77142891545934,
 'attention_ms': 14.472623040356973,
 'matched_rows': 254.0}